# 睡眠剥夺 scRNA-seq: 从数据下载到预处理  
## 在 Google Colab 中按顺序运行每个单元格 (Ctrl+F9 = 运行全部)  
**重要**: 菜单栏 → 修改 → 笔记本设置 → 硬件加速选 GPU (T4)

In [ ]:
# ==========================================================
# 第 1 步: 安装依赖 (约 2 分钟)
# ==========================================================
!pip install -q scanpy pandas numpy matplotlib seaborn scipy leidenalg anndata
!pip install -q GEOparse

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

print(f'scanpy: {sc.__version__}')
sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=True)
sc.logging.print_header()

In [ ]:
# ==========================================================
# 第 2 步: 挂载 Google Drive (保存结果用)
# ==========================================================
from google.colab import drive
drive.mount('/content/drive')

# 创建项目文件夹
!mkdir -p /content/drive/MyDrive/sleep_deprivation_project/data
!mkdir -p /content/drive/MyDrive/sleep_deprivation_project/results
DATA_DIR = '/content/drive/MyDrive/sleep_deprivation_project/data'
RESULT_DIR = '/content/drive/MyDrive/sleep_deprivation_project/results'
print('Google Drive 已挂载，项目文件夹已创建')

In [ ]:
# ==========================================================
# 第 3 步: 下载 GSE137665 (小鼠睡眠剥夺 3 脑区 scRNA-seq)
# 数据来源: GEO (Colab 云端不受国内网络限制)
# ==========================================================

import GEOparse

# 下载 GEO 元数据
gse = GEOparse.get_GEO(geo='GSE137665', destdir='/content/data', silent=True)
print(f'Title: {gse.metadata["title"][0]}')
print(f'Samples: {len(gse.gsms)}')

# 显示每个样本的信息
for gsm_name, gsm in gse.gsms.items():
    print(f'\n{gsm_name}:')
    for key, value in list(gsm.metadata.items())[:5]:
        print(f'  {key}: {value}')

In [ ]:
# ==========================================================
# 第 4 步: 下载样本的表达矩阵
# scRNA-seq 数据通常以 supplementary files 形式提供
# 可能是 .h5, .mtx, .h5ad 或压缩包格式
# ==========================================================

# 检查补充文件
for gsm_name, gsm in gse.gsms.items():
    suppl = gsm.metadata.get('supplementary_file', [])
    if isinstance(suppl, str):
        suppl = [suppl]
    print(f'{gsm_name}: {len(suppl)} supplementary files')
    for f in suppl:
        print(f'  {f}')

# 下载第一个文件的示例（调整文件类型过滤）
from urllib.request import urlretrieve

def download_supplementary(gse, file_pattern='.h5'):
    """下载匹配 pattern 的补充文件"""
    os.makedirs('/content/data/GSE137665', exist_ok=True)
    
    for gsm_name, gsm in gse.gsms.items():
        suppl = gsm.metadata.get('supplementary_file', [])
        if isinstance(suppl, str):
            suppl = [suppl]
        for url in suppl:
            if file_pattern in url:
                fname = os.path.basename(url)
                path = f'/content/data/GSE137665/{fname}'
                if not os.path.exists(path):
                    print(f'Downloading {fname}...')
                    urlretrieve(url, path)
                else:
                    print(f'{fname} already downloaded')

# 尝试下载（GSE137665 的具体文件格式需要运行后才知道）
# download_supplementary(gse, '.h5')
print('样本信息检查完成，请根据输出调整下载参数')

In [ ]:
# ==========================================================
# 第 5 步: 加载数据到 AnnData（根据实际文件格式选择）
# ==========================================================

# 方式 A: 如果是 10x Genomics h5 文件
# adata = sc.read_10x_h5('/content/data/GSE137665/sample.h5')

# 方式 B: 如果是 mtx 格式 (10x CellRanger 输出)
# adata = sc.read_10x_mtx('/content/data/GSE137665/filtered_feature_bc_matrix/')

# 方式 C: 如果是 h5ad 格式
# adata = sc.read_h5ad('/content/data/GSE137665/sample.h5ad')

# 方式 D: 如果有批次，逐个加载后合并
# adata_list = [sc.read_10x_h5(f) for f in file_list]
# adata = adata_list[0].concatenate(adata_list[1:], batch_categories=['ctrl','sd'])

# 方式 E: 如果数据在 GEO 中是 count CSV/TSV
# df = pd.read_csv('/content/data/GSE137665/counts.csv', index_col=0)
# adata = sc.AnnData(df.T)

print('请根据上一步下载的文件格式，取消注释对应的加载代码')
# print(adata)

In [ ]:
# ==========================================================
# 第 6 步: QC + 标准化 + 降维 + 聚类
# ==========================================================

def basic_preprocessing(adata, batch_key=None):
    # 质控指标
    adata.var['mt'] = adata.var_names.str.startswith('mt-')
    adata.var['ribo'] = adata.var_names.str.startswith(('Rps', 'Rpl'))
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt', 'ribo'], percent_top=None, log1p=False, inplace=True)
    
    # QC 过滤
    print(f'Before filtering: {adata.n_obs} cells, {adata.n_vars} genes')
    sc.pp.filter_cells(adata, min_genes=200)
    sc.pp.filter_genes(adata, min_cells=3)
    adata = adata[adata.obs.n_genes_by_counts < 5000, :]
    adata = adata[adata.obs.pct_counts_mt < 20, :]
    print(f'After filtering: {adata.n_obs} cells, {adata.n_vars} genes')
    
    # 标准化 + 高变基因
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=2000, batch_key=batch_key)
    adata.raw = adata
    adata = adata[:, adata.var.highly_variable]
    
    # 回归线粒体比例 + 缩放
    sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt'])
    sc.pp.scale(adata, max_value=10)
    
    # PCA + 邻居 + UMAP + 聚类
    sc.tl.pca(adata, svd_solver='arpack', n_comps=30)
    sc.pp.neighbors(adata, n_pcs=20, n_neighbors=15)
    sc.tl.umap(adata, min_dist=0.3)
    sc.tl.leiden(adata, resolution=0.5)
    
    return adata

# adata = basic_preprocessing(adata)
# print(adata)
print('预处理函数已定义，加载数据后即可运行')

In [ ]:
# ==========================================================
# 第 7 步: 找 marker 基因 + 保存处理后的数据
# ==========================================================

# marker_genes
# sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon')

# 保存为 h5ad 文件
# adata.write('/content/drive/MyDrive/sleep_deprivation_project/data/GSE137665_processed.h5ad')
# print('处理后的数据已保存到 Google Drive')

# 导出表达矩阵为 CSV（可以下载到本地用 R 分析）
# expr_df = adata.to_df().T
# expr_df.to_csv('/content/drive/MyDrive/sleep_deprivation_project/data/expression_matrix.csv')
# print('Expression matrix saved as CSV')

# 下载到电脑
# from google.colab import files
# files.download('/content/drive/MyDrive/sleep_deprivation_project/data/GSE137665_processed.h5ad')

print('请加载数据后取消注释以上代码')

In [ ]:
# ==========================================================
# 第 8 步: 初步可视化 (快速查看数据质量)
# ==========================================================

# UMAP
# sc.pl.umap(adata, color=['leiden'], legend_loc='on data', 
#            title='Leiden Clusters', frameon=True, show=True)

# QC 指标分布
# fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# sc.pl.violin(adata, 'n_genes_by_counts', ax=axes[0], show=False)
# sc.pl.violin(adata, 'total_counts', ax=axes[1], show=False)
# sc.pl.violin(adata, 'pct_counts_mt', ax=axes[2], show=False)
# plt.tight_layout()
# plt.savefig('/content/drive/MyDrive/sleep_deprivation_project/results/QC_plots.png', dpi=150, bbox_inches='tight')
# plt.show()

print('可视化代码准备就绪')